In [ ]:

! pip install unsloth datasets transformers

In [ ]:
import os
import pandas as pds
import unsloth
import torch
import transformers
import datasets
import gc
import json
import time
from pathlib import Path


In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from unsloth import FastLanguageModel, is_bf16_supported

print("done")

In [ ]:
from google.colab import drive
DATA_DIR = drive.mount('/content/drive')

In [ ]:
from datasets import load_dataset


trn_data = "/content/drive/MyDrive/FINE-TUNE/DATASET/dataset_train.jsonl"
tst_data = "/content/drive/MyDrive/FINE-TUNE/DATASET/dataset_test.jsonl"
vld_data = "/content/drive/MyDrive/FINE-TUNE/DATASET/dataset_validation.jsonl"

data_files = {
    "train": trn_data,
    "validation": vld_data,
    "test": tst_data,
}
datasets = load_dataset("json", data_files=data_files)

print(datasets)

In [ ]:

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
MAX_SEQ_LENGTH= 2048


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True
)

In [ ]:
#  saving some examples from base model to compare it with fine-tuned one

from transformers import pipeline

# 1. Extract the 3 splits from your `datasets` DatasetDict
train_dataset = datasets["train"]
val_dataset   = datasets["validation"]
test_dataset  = datasets["test"]

# 2. Pick 5 unseen test rows for our Before vs. After comparison
eval_test_slice = test_dataset.select(range(5))

def get_answers(model, tokenizer, test_slice):
    FastLanguageModel.for_inference(model)
    answers = []

    for r in test_slice:
        # Pass conversation history excluding the final assistant target
        prompt_messages = [m for m in r["messages"] if m["role"] != "assistant"]

        inputs = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        ).to("cuda")

        with torch.inference_mode():
            outputs = model.generate(
                input_ids=inputs,
                max_new_tokens=256,
                temperature=0.2,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        # Slice off prompt tokens to keep only the newly generated response
        response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        answers.append(response.strip())

    return answers

print("Generating Base Model baseline answers...")
base_model_answers = get_answers(model, tokenizer,eval_test_slice)
print(f"\nSample 1 Base Answer:\n{base_model_answers[0][:250]}...")

In [ ]:
#  Adding LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout=0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora=False,
    loftq_config=None

)

In [ ]:
# 1. Switch model back into training mode
FastLanguageModel.for_training(model)

# 2. Convert the 'messages' list of dicts into a flat Qwen3 ChatML string
def format_chat_prompts(batch):
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,               # Keep as readable text; SFTTrainer tokenizes it later
            add_generation_prompt=False   # False because the assistant's answer is already inside convo!
        )
        for convo in batch["messages"]
    ]
    return {"text": texts}

train_dataset = train_dataset.map(
    format_chat_prompts,
    batched=True,
    remove_columns=train_dataset.column_names,
)
val_dataset = val_dataset.map(
    format_chat_prompts,
    batched=True,
    remove_columns=train_dataset.column_names,
)

# 4. Check how many parameters we are actually training & preview Row 0
model.print_trainable_parameters()
print("\n--- FORMATTED TRAINING TEXT PREVIEW ---")
print(train_dataset[0]["text"][:350])

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers.trainer_utils import get_last_checkpoint

CHECKPOINT_DIR = "/content/drive/MyDrive/FINE-TUNE/qwen3_4b_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

training_args = SFTConfig(
    output_dir = CHECKPOINT_DIR,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,      # 2048

    # 1. Your Tuned T4 Batching (Effective Batch Size = 6 * 2 = 12)
    packing = True,
    per_device_train_batch_size = 2,      
    gradient_accumulation_steps = 6,      
    dataloader_num_workers = 1,           
    dataloader_pin_memory = True,

    # 2. Learning Schedule & 8-Bit Optimizer
    num_train_epochs = 3,
    learning_rate = 2e-4,
    warmup_ratio = 0.05,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",

    # 3. Precision (fp16 on Tesla T4)
    fp16 = not is_bf16_supported(),
    bf16 = is_bf16_supported(),

    # 4. Logging, Validation & Google Drive Checkpoints
    logging_steps = 5,
    eval_strategy = "steps",
    eval_steps = 50,
    save_strategy = "steps",
    save_steps = 50,
    save_total_limit = 2,
    seed = 42,
    report_to = "none",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    return_tensors="pt",
    padding=True,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    args = training_args,
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# Auto-resume from Google Drive if Colab ever disconnects
last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)
if last_checkpoint is not None:
    print(f"🔄 Resuming from checkpoint: {last_checkpoint}")
    trainer_stats = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("🚀 Starting fresh packed training run...")
    trainer_stats = trainer.train()

In [ ]:
from IPython.display import display

# 1. Generate answers using the newly trained model (automatically switches to inference mode!)
print("Generating Fine-Tuned Model answers on the same 5 unseen test questions...")
fine_tuned_answers = get_answers(model, tokenizer,eval_test_slice)

# Dynamically locate user prompt (skips system message at index 0)
questions = [
    next((m["content"] for m in r["messages"] if m["role"] == "user"), "")
    for r in eval_test_slice
]

# Dynamically locate target assistant answer
target_ground_truth = [
    next((m["content"] for m in reversed(r["messages"]) if m["role"] == "assistant"), "")
    for r in eval_test_slice
]

# 3. Build a clean side-by-side comparison table (using `pds` from your imports!)
pds.set_option("display.max_colwidth", None)  # Prevents Pandas from cutting off long text with "..."

comparison_df = pds.DataFrame({
    "Question": questions,
    "Target Answer (Dataset)": target_ground_truth,
    "Base Model (Before)": base_model_answers,
    "Fine-Tuned (After)": fine_tuned_answers,
})

print("\n--- QUALITATIVE COMPARISON: BEFORE VS. AFTER ---")
display(comparison_df)

In [ ]:
! pip install evaluate
import evaluate

print("\n--- CALCULATING QUANTITATIVE ROUGE SCORES ---")
rouge = evaluate.load("rouge")

# 1. Compute ROUGE for Base Model vs Target Dataset
base_rouge = rouge.compute(
    predictions=base_model_answers,
    references=target_ground_truth
)

# 2. Compute ROUGE for Fine-Tuned Model vs Target Dataset
ft_rouge = rouge.compute(
    predictions=fine_tuned_answers,
    references=target_ground_truth
)

# 3. Build a clean score comparison table using Pandas
score_df = pds.DataFrame({
    "Metric": ["ROUGE-1 (Keywords)", "ROUGE-2 (Phrases)", "ROUGE-L (Structure)"],
    "Base Model": [
        f"{base_rouge['rouge1'] * 100:.2f}%",
        f"{base_rouge['rouge2'] * 100:.2f}%",
        f"{base_rouge['rougeL'] * 100:.2f}%"
    ],
    "Fine-Tuned": [
        f"{ft_rouge['rouge1'] * 100:.2f}%",
        f"{ft_rouge['rouge2'] * 100:.2f}%",
        f"{ft_rouge['rougeL'] * 100:.2f}%"
    ],
    "Improvement": [
        f"{(ft_rouge['rouge1'] - base_rouge['rouge1']) * 100:+.2f}%",
        f"{(ft_rouge['rouge2'] - base_rouge['rouge2']) * 100:+.2f}%",
        f"{(ft_rouge['rougeL'] - base_rouge['rougeL']) * 100:+.2f}%",
    ]
})

display(score_df)

# saving adapters to the google drive so that we can use the adapters later
ADAPTER_SAVE_PATH = "/content/drive/MyDrive/FINE-TUNE/qwen3_4b_lora_adapter"

print(f"\nSaving final LoRA adapter weights to: {ADAPTER_SAVE_PATH}")
model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
print("✅ Training pipeline complete! LoRA adapters saved securely to Google Drive.")

In [ ]:
import shutil

# 1. Export to Colab's fast local SSD instead of Google Drive
LOCAL_GGUF_PATH = "/content/Qwen_PRIME4B_AI_gguf"

model.save_pretrained_gguf(
    LOCAL_GGUF_PATH,
    tokenizer,
    quantization_method = "q4_k_m"
)

# 2. Once finished, copy the finished GGUF folder safely to Google Drive
DRIVE_DEST = "/content/drive/MyDrive/FINE-TUNE/Qwen_PRIME4B_AI_gguf"
shutil.copytree(LOCAL_GGUF_PATH, DRIVE_DEST, dirs_exist_ok=True)
print("✅ GGUF successfully built locally and copied to Google Drive!")

In [ ]:
import shutil

# Copy from the correct local folder where Unsloth saved the .gguf file
shutil.copytree("/content/Qwen_PRIME4B_AI_gguf_gguf", "/content/drive/MyDrive/FINE-TUNE/Qwen_PRIME4B_AI_gguf", dirs_exist_ok=True)
print("✅ GGUF model successfully copied to Google Drive!")